# 호텔 챗봇

이 노트북에서는 호텔 챗봇 문제를 다룹니다. 어떻게 하면 호텔 프론트 데스크에서 더 나은 고객 서비스를 제공할 수 있을까요? 간단한 방법 중 하나는 호텔에서 경험할 수 있는 일들에 관한 간단한 질문에 대답할 수 있는 FAQ 채팅 로봇을 갖는 것입니다. 챗봇을 갖는 것에는 많은 장점이 있습니다.

1. 호텔의 매력을 높이고 정보 처리량을 늘립니다.
2. 호텔에 대한 질문을 데이터 테이블 형식으로 수집할 수 있는 방법을 만듭니다.

이 노트북에서는 두 가지 모델을 살펴볼 것입니다. doc2vec 모델과 코사인 유사성 모델입니다.


## 1. 코사인 유사성을 이용한 챗봇 기능

In [1]:
import os
import nltk # 텍스트 데이터를 처리
import numpy as np
import random
import string # 표준 파이썬 문자열을 처리
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
filepath=os.getcwd()+r'//[Dataset] Module27 (ans).txt'
corpus=open(filepath,'r',errors = 'ignore')
raw_data_ans=corpus.read()
print (raw_data_ans)

print('-'*90)
filepath=os.getcwd()+'//[Dataset] Module27(ques).txt'
corpus=open(filepath,'r',errors = 'ignore')
raw_data=corpus.read()
print (raw_data)

200$ per night is the price for a basic suite.

This establishment was constructed and inaugurated by John S. on the 23rd of September 1965.

Breakfast is served from 7 AM to 10 AM.

The breakfast menu is decided as per the head chefs decision on the night before; kindly contact hotel staff for information about the menu on the night before. 

The Vance Hotel is a singular establishment designed and constructed by Lindsey Vance in 1949; it has hosted many dignitaries and government officials over the years.

There are 43 rooms in this hotel including one pent house suite.

We offer 4 types of rooms: basic, mid-level, premium and penthouse.

This hotel is called Vance Hotel.

Yes, room service is available 24 hrs.

To call room service, please dial '0' using the phone in your room.

Yes we have one restaurant currently called 'Rouge'.

There are 12 floors in the hotel.

300$ per night is the standard price for a mid-level suite.

500$ per night is the price for a premium suite.

Yes, we

####  소문자로 변환

모든 텍스트를 먼저 소문자로 변환합니다. 결과가 완료되면 검사해야 합니다.

In [3]:
# raw_data를 소문자로 바꾸고 다시 raw_data로 저장한 후 출력해 봅니다.
# your code here
# raw_data를 소문자로 바꾸고 다시 raw_data로 저장한 후 출력해 봅니다.
raw_data = raw_data.lower()
print(raw_data)


what is the price of one night stay in basic suite?

how old is this establishment?

what time is breakfast served?

what is the breakfast menu?

what is the history behind this hotel or establishment?

how many rooms are there in this hotel?

what are the types of rooms or suites offered by the hotel?

what is the name of the hotel?

is room service served 24 hours?

how do i call for room service?

are there any restaurants in the hotel?

how many floors are there in the hotel?

what is the price of one night stay at the mid-level suite?

what is the price of one night stay at the premium suite?

do you have tuxedo services?

do you have a laundry service?

what time do the restaurants open for dinner?

what are the near by tourist attractions?

is there a spa in the hotel?

is there anywhere i can get a massage?

what time is the check in?

what time is the check out?

do you offer handicapped rooms?

is parking available at the hotel?

can i reserve a parking lot?

what are the rec

####  세분화, 표제어 추출,  단어 토큰화

모듈 26 코사인 유사성 노트북에서 배운 방법을 참조하여 데이터를 세분화, 토큰화, 표제어 추출로 전처리 진행합니다.

In [4]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')
# raw_data를 sent_tokenize() 함수로 문장으로 구분한 후 sent_tokens에 저장하세요.
# sent_tokens를 출력합니다.
# your code here
sent_tokens = nltk.sent_tokenize(raw_data)
print(sent_tokens)

['what is the price of one night stay in basic suite?', 'how old is this establishment?', 'what time is breakfast served?', 'what is the breakfast menu?', 'what is the history behind this hotel or establishment?', 'how many rooms are there in this hotel?', 'what are the types of rooms or suites offered by the hotel?', 'what is the name of the hotel?', 'is room service served 24 hours?', 'how do i call for room service?', 'are there any restaurants in the hotel?', 'how many floors are there in the hotel?', 'what is the price of one night stay at the mid-level suite?', 'what is the price of one night stay at the premium suite?', 'do you have tuxedo services?', 'do you have a laundry service?', 'what time do the restaurants open for dinner?', 'what are the near by tourist attractions?', 'is there a spa in the hotel?', 'is there anywhere i can get a massage?', 'what time is the check in?', 'what time is the check out?', 'do you offer handicapped rooms?', 'is parking available at the hotel?

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
# raw_data_ans를 sent_tokenize() 함수로 문장으로 구분한 후 sent_tokens_ans에 저장하세요.
# sent_tokens_ans를 출력합니다.
# your code here
sent_tokens_ans = nltk.sent_tokenize(raw_data_ans)
print(sent_tokens_ans)

['200$ per night is the price for a basic suite.', 'This establishment was constructed and inaugurated by John S. on the 23rd of September 1965.', 'Breakfast is served from 7 AM to 10 AM.', 'The breakfast menu is decided as per the head chefs decision on the night before; kindly contact hotel staff for information about the menu on the night before.', 'The Vance Hotel is a singular establishment designed and constructed by Lindsey Vance in 1949; it has hosted many dignitaries and government officials over the years.', 'There are 43 rooms in this hotel including one pent house suite.', 'We offer 4 types of rooms: basic, mid-level, premium and penthouse.', 'This hotel is called Vance Hotel.', 'Yes, room service is available 24 hrs.', "To call room service, please dial '0' using the phone in your room.", "Yes we have one restaurant currently called 'Rouge'.", 'There are 12 floors in the hotel.', '300$ per night is the standard price for a mid-level suite.', '500$ per night is the price fo

#### 질문과 답 쌍으로 만들어진 Dictionary를 만듭니다. #### 

In [6]:
# sent_tokens가 key가 되고 sent_tokens_ans가 value가 되는 Dictionary를 만들고 res에 저장합니다.
# your code here
res = dict(zip(sent_tokens, sent_tokens_ans))
# res를 출력합니다.
print(res)


{'what is the price of one night stay in basic suite?': '200$ per night is the price for a basic suite.', 'how old is this establishment?': 'This establishment was constructed and inaugurated by John S. on the 23rd of September 1965.', 'what time is breakfast served?': 'Breakfast is served from 7 AM to 10 AM.', 'what is the breakfast menu?': 'The breakfast menu is decided as per the head chefs decision on the night before; kindly contact hotel staff for information about the menu on the night before.', 'what is the history behind this hotel or establishment?': 'The Vance Hotel is a singular establishment designed and constructed by Lindsey Vance in 1949; it has hosted many dignitaries and government officials over the years.', 'how many rooms are there in this hotel?': 'There are 43 rooms in this hotel including one pent house suite.', 'what are the types of rooms or suites offered by the hotel?': 'We offer 4 types of rooms: basic, mid-level, premium and penthouse.', 'what is the name 

#### Module 26을 참고해서 greeting 함수를 작성하세요. ####

In [7]:
import random

GREETING_INPUTS = ["hello", "hi", "greetings", "sup", "what's up","hey", "hey there"]
GREETING_RESPONSES = ["hi", "hey", "*nods*", "hi there", "hello", "I am glad! You are talking to me"]

def greeting(sentence):
    # 입력된 문장을 단어 단위로 나누어 확인합니다.
    for word in sentence.split():
        # 단어를 소문자로 변환하여 GREETING_INPUTS에 있는지 확인합니다.
        if word.lower() in GREETING_INPUTS:
            # 존재한다면 GREETING_RESPONSES 중에서 무작위로 하나를 반환합니다.
            return random.choice(GREETING_RESPONSES)

greeting('hi')


'hey'

#### Module 26을 참고해서 LemTokens()와 LemNormalize() 함수를 그대로 작성하세요. ####

In [8]:
from nltk import pos_tag
import nltk
nltk.download('averaged_perceptron_tagger_eng')

def get_wordnet_pos(tag):
    # 태그를 WordNet POS ('n', 'v', 'a', 'r')로 변환
    if tag.startswith('J'):    # 형용사
        return 'a'
    elif tag.startswith('V'):  # 동사
        return 'v'
    elif tag.startswith('R'):  # 부사
        return 'r'
    else:
        return 'n'             # 명사

lemmer = nltk.stem.WordNetLemmatizer()

def LemTokens(tokens):
    # 1. 토큰들의 품사를 태깅합니다. (word, tag) 형태의 튜플 리스트 반환
    pos_tags = pos_tag(tokens)
    
    # 2. 각 단어별로 WordNet POS 태그로 변환한 뒤 표제어(Lemma)를 추출하여 리스트로 반환합니다.
    return [lemmer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [9]:
import string
import nltk

# 구두점을 None으로 맵핑하여 제거하기 위한 딕셔너리
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)

def LemNormalize(text):
    # 1. 텍스트를 소문자로 변환(lower)하고, 문장 부호를 제거합니다(translate).
    # 2. 전처리된 텍스트를 단어 단위로 토큰화합니다(word_tokenize).
    # 3. 앞서 정의한 LemTokens 함수에 전달하여 표제어(Lemma) 리스트를 반환합니다.
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))


챗봇의 기능은 챗봇을 실행하기 위한 루프를 만듦으로써 이루어집니다. 아래 기능을 살펴봅니다. 함수의 각 줄은 중요한 단계를 수행하기 위해 다른 함수를 호출하기 때문에 중요합니다. 'response' 함수는 챗봇이 어떻게 행동하는지에 대한 작동 방식을 담당합니다.

In [10]:
# 아래 response() 함수는 Module 26의 Cosine Similarity에 나오는 response() 함수와 거의 동일합니다.
# 전체 Dictionary를 TFIDF로 벡터화를 진행하고 
# 입력한 문장과 가장 근접한 sentence를 sent_tokens에서 찾으면 그에 해당되는 답변을 출력하도록 코드를 수정해 주세요.
# return 시에는 답변과 유사도를 순서대로 전달합니다.

def response(user_response):
    sent_tokens.append(user_response)
    
    TfidfVec = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english')
    tfidf = TfidfVec.fit_transform(sent_tokens)
    
    vals = cosine_similarity(tfidf[-1], tfidf)
    idx = vals.argsort()[0][-2]
    flat = vals.flatten()
    flat.sort()
    req_tfidf = flat[-2]
    
    if(req_tfidf == 0):
        bot_response = "I am sorry! I don't understand you"
        return bot_response, req_tfidf
    else:
        # sent_tokens에서 해당 질문을 찾고, res 딕셔너리를 사용하여 답변을 매핑합니다.
        bot_response = res[sent_tokens[idx]]
        return bot_response, req_tfidf


In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

flag=True
print("Jane: My name is Jane. I will answer your queries about this hotel. If you want to exit, type Bye!")
while(flag==True):
    user_response = input()
    user_response=user_response.lower()
    if(user_response!='bye'):
        if(user_response=='thanks' or user_response=='thank you' ):
            flag=False
            print("Jane: You are welcome..")
        else:
            if(greeting(user_response)!=None):
                print("Jane: "+greeting(user_response))

            else:
                print("Jane: ",end="")
                resp = response(user_response)
                print(resp[0], )
                sent_tokens.remove(user_response)
                print(f'(With similarity of {resp[1]:.2f})')

    else:
        flag=False
        print("Jane: Bye! take care..")


Jane: My name is Jane. I will answer your queries about this hotel. If you want to exit, type Bye!
Jane: This hotel is called Vance Hotel.
(With similarity of 0.47)
Jane: I am sorry! I don't understand you
(With similarity of 0.00)
Jane: I am sorry! I don't understand you
(With similarity of 0.00)
Jane: I am sorry! I don't understand you
(With similarity of 0.00)
Jane: I am sorry! I don't understand you
(With similarity of 0.00)


## 2. Doc2vec를 이용한 챗봇 기능

챗봇을 만들기 위한 한 가지 유형의 모델을 더 다룰 것입니다. 코사인 유사성 모델 알고리즘은 두 문장 사이의 유사성을 찾는데 사용됩니다. 하지만 이제 신경망을 사용해서 이 문제를 해결해 보려합니다. 살펴보도록 하겠습니다.

Doc2Vec은 기본적으로 문서에서 벡터를 생성하는 신경망 기반 모델입니다. Doc2vec을 이해하기 위해서는 word2vec도 이해해야 합니다.

#### word2vec란?
이는 삽입 단어를 생성하는 모델이며, 여기서는 텍스트의 큰 말뭉치를 입력으로 받고 일반적으로 수백 개 차원의 벡터 공간을 생성합니다.

2013년 9월과 10월 사이에 구글의 연구팀에 의해 두 개의 논문에 소개되었습니다. Word2Vec의 기본 가정은 유사한 맥락을 공유하는 두 단어가 유사한 의미를 공유하며 결과적으로 모델에서 유사한 벡터 표현을 공유한다는 것입니다.

예를 들어, "은행", "화폐", "계좌"는 "달러", "대출", "신용"과 같은 유사한 주변 단어와 함께 종종 사용되며, 따라서 Word2Vec에 따르면 이들은 유사한 벡터 표현을 공유합니다.

#### doc2vec 란?

doc2vec의 목적은 말뭉치의 모든 단어에 대한 특징 벡터를 계산하는 word2vec와 달리 문장/단락/문서의 수치 표현을 생성하는 것입니다. doc2vec은 말뭉치의 모든 문서에 대한 특징 벡터를 계산합니다. doc2vec에 의해 생성된 벡터는 문장/단락/문서 간의 유사성 찾기와 같은 작업에 사용될 수 있습니다.

<strong> doc2vec의 속성을 사용하여 우리만의 유사성 모델을 만들 것입니다.</strong>


관련 라이브러리를 가져오는 것으로 시작하겠습니다.

In [1]:
!pip install gensim

In [2]:
from gensim.models.doc2vec import Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
import os
import nltk

ImportError: cannot import name 'triu' from 'scipy.linalg' (c:\Users\User\anaconda3\Lib\site-packages\scipy\linalg\__init__.py)

In [ ]:
filepath=os.getcwd()+r'//[Dataset] Module27 (ans).txt'
corpus=open(filepath,'r',errors = 'ignore')
raw_data_ans=corpus.read()
sent_tokens_ans = nltk.sent_tokenize(raw_data_ans)

filepath=os.getcwd()+'//[Dataset] Module27(ques).txt'
corpus=open(filepath,'r',errors = 'ignore')
raw_data=corpus.read()
sent_tokens = nltk.sent_tokenize(raw_data)

## 2. 챗봇에 지식 기반 추가

챗봇의 대화 능력은 사용할 수 있는 데이터에 의해 정의됩니다. Ques.txt 파일과 ans.txt 파일에서 질문과 답변을 살펴보십시오. 이 챗봇은 기본적으로 질문에 대하여 질문 은행과 코사인 유사성을 확인하여 답을 찾으려고 노력할 것입니다.

각 문장(문서)을 토큰화하고, 이를 고유한 태그와 함께 TaggedDocument로 변환하여 Doc2Vec 모델의 입력 데이터로 사용하기 위함입니다.


In [21]:
tagged_data = [TaggedDocument(words=word_tokenize(_d.lower()), tags=[str(i)]) for i, _d in enumerate(sent_tokens)]

In [31]:
tagged_data

[TaggedDocument(words=['what', 'is', 'the', 'price', 'of', 'one', 'night', 'stay', 'in', 'basic', 'suite', '?'], tags=['0']),
 TaggedDocument(words=['how', 'old', 'is', 'this', 'establishment', '?'], tags=['1']),
 TaggedDocument(words=['what', 'time', 'is', 'breakfast', 'served', '?'], tags=['2']),
 TaggedDocument(words=['what', 'is', 'the', 'breakfast', 'menu', '?'], tags=['3']),
 TaggedDocument(words=['what', 'is', 'the', 'history', 'behind', 'this', 'hotel', 'or', 'establishment', '?'], tags=['4']),
 TaggedDocument(words=['how', 'many', 'rooms', 'are', 'there', 'in', 'this', 'hotel', '?'], tags=['5']),
 TaggedDocument(words=['what', 'are', 'the', 'types', 'of', 'rooms', 'or', 'suites', 'offered', 'by', 'the', 'hotel', '?'], tags=['6']),
 TaggedDocument(words=['what', 'is', 'the', 'name', 'of', 'the', 'hotel', '?'], tags=['7']),
 TaggedDocument(words=['is', 'room', 'service', 'served', '24', 'hours', '?'], tags=['8']),
 TaggedDocument(words=['how', 'do', 'i', 'call', 'for', 'room', '

In [32]:
max_epochs = 100
vec_size = 20 # 벡터가 더 크려면 이것을 증가시키세요. 이것은 더 많은 차이를 의미합니다.
alpha = 0.025

다음 단계는 모델을 훈련시키는 것입니다. 이전처럼 훈련 과정을 실행하기 위해 `model.train` 함수를 사용하게 될 것입니다. Doc2vec 모델을 훈련하는 방법에 대한 자세한 정보는 [문서](https://radimrehurek.com/gensim/models/doc2vec.html) 를 참조하십시오.

In [33]:
model = Doc2Vec(vector_size=vec_size,
                alpha=alpha,      # learning rate
                min_alpha=0.00025,
                min_count=1,      # 단어 빈도 조정
                dm=1) # dm=0 → Word2Vec의 Skip-gram / dm=1 → CBOW 방식과 유사.

model.build_vocab(tagged_data)    # 학습할 때 사용할 어휘(vocabulary)를 구축
print('Model.corpus_count :', model.corpus_count)

for epoch in range(max_epochs):
    print('iteration {0}'.format(epoch))
    model.train(tagged_data,
                total_examples=model.corpus_count,
                epochs=100)
    model.alpha -= 0.0002
    model.min_alpha = model.alpha

model.save("d2v.model")
print("Model Saved")


Model.corpus_count : 47
iteration 0
iteration 1
iteration 2
iteration 3
iteration 4
iteration 5
iteration 6
iteration 7
iteration 8
iteration 9
iteration 10
iteration 11
iteration 12
iteration 13
iteration 14
iteration 15
iteration 16
iteration 17
iteration 18
iteration 19
iteration 20
iteration 21
iteration 22
iteration 23
iteration 24
iteration 25
iteration 26
iteration 27
iteration 28
iteration 29
iteration 30
iteration 31
iteration 32
iteration 33
iteration 34
iteration 35
iteration 36
iteration 37
iteration 38
iteration 39
iteration 40
iteration 41
iteration 42
iteration 43
iteration 44
iteration 45
iteration 46
iteration 47
iteration 48
iteration 49
iteration 50
iteration 51
iteration 52
iteration 53
iteration 54
iteration 55
iteration 56
iteration 57
iteration 58
iteration 59
iteration 60
iteration 61
iteration 62
iteration 63
iteration 64
iteration 65
iteration 66
iteration 67
iteration 68
iteration 69
iteration 70
iteration 71
iteration 72
iteration 73
iteration 74
iteration 7

### doc2vec 모델 평가

In [34]:
from gensim.models.doc2vec import Doc2Vec
model= Doc2Vec.load("d2v.model")

In [35]:
test_data = word_tokenize("How much is the price?".lower())

`model.infer_vector` 함수를 사용하여 문서와 관련된 벡터를 추론할 수 있습니다. 그런 다음`most_similar` 함수를 사용하여 우리가 만든 벡터와 가장 유사한 벡터를 찾을 수 있습니다. 결과는 어떻습니까?

In [37]:
v1 = model.infer_vector(test_data)
print("V1_infer", v1)

V1_infer [-0.07546428  0.25868264  0.426788    0.43974108 -0.30752054 -0.0921553
 -0.19883995  0.43241897 -0.80354553 -0.01300353  0.22424833  0.09113787
 -0.0487121   0.00281997 -0.03348381  0.06258876  0.3049244  -0.0122046
  0.01522969 -0.32715365]


In [38]:
similar_doc = model.dv.most_similar(positive = [v1], topn = 4) #positive is an attribute that shows positive correlation first followed by the correlation value
print(similar_doc)

[('31', 0.7440255284309387), ('29', 0.6993741989135742), ('25', 0.6869140863418579), ('40', 0.6561254858970642)]


C:\Users\user\AppData\Local\Temp\ipykernel_8676\1502821758.py:1: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  similar_doc = model.docvecs.most_similar(positive = [v1], topn = 4) #positive is an attribute that shows positive correlation first followed by the correlation value


In [39]:
num,_ = similar_doc[0]
num = int(num)
tagged_data[num]

TaggedDocument(words=['why', 'do', 'the', 'costs', 'vary', 'from', 'day', 'to', 'day', '?'], tags=['31'])

In [30]:
sent_tokens_ans = nltk.sent_tokenize(raw_data_ans)

print(sent_tokens_ans[num])

The rates may vary as per availability and demand.


이전 코드 블록의 출력에서 이 모델이 생각했던 것 만큼 효과적이지 않다는 것이 분명합니다. 이 모델이 성공하기를 기대했지만 실패한 이유가 있습니까?

자세한 내용을 보려면 이 [링크](https://stackoverflow.com/questions/58206571/doc2vec-find-the-similar-sentence) 를 클릭하십시오. 이 문제의 요지는 다음과 같습니다.

> Doc2Vec는 장난감 크기(toy-size)의 데이터셋에서 좋은 결과를 얻을 수 없으므로, 더 많은 데이터를 사용하기 전까지는 의미 있는 결과를 기대해서는 안 됩니다.

In [ ]:
GREETING_INPUTS = ["hello", "hi", "greetings", "sup", "what's up","hey", "hey there"]
GREETING_RESPONSES = ["hi", "hey", "*nods*", "hi there", "hello", "I am glad! You are talking to me"]

def greeting(sentence):
# your code here
    for word in sentence.split():
        if word.lower() in GREETING_INPUTS:
            return random.choice(GREETING_RESPONSES)

In [ ]:
def doc2vec_response(user_response) :
    user_tokens = word_tokenize(user_response.lower())
    v1 = model.infer_vector(user_tokens)
    similar_doc = model.dv.most_similar(positive = [v1], topn = 1)
    #print(similar_doc)
    return sent_tokens_ans[int(similar_doc[0][0])], similar_doc[0][1]

In [ ]:
import warnings
import random

warnings.filterwarnings("ignore", category=UserWarning)

flag=True
print("Jane: My name is Jane. I will answer your queries about this hotel. If you want to exit, type Bye!")
while(flag==True):
    user_response = input()
    user_response=user_response.lower()
    if(user_response!='bye'):
        if(user_response=='thanks' or user_response=='thank you' ):
            flag=False
            print("Jane: You are welcome..")
        else:
            if(greeting(user_response)!=None):
                print("Jane: "+greeting(user_response))

            else:
                print("Jane: ",end="")

                resp, similarity = doc2vec_response(user_response)
                print(resp)
                print(f'(With similarity of {similarity:.2f})')

    else:
        flag=False
        print("Jane: Bye! take care..")
